In [ ]:
from transformers import AutoModelForCausalLM, AutoConfig
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2-xl")

config = AutoConfig.from_pretrained("openai-community/gpt2-xl")

print(config.model_type)




In [ ]:
from glob import glob
import os

input = "/users/rh/.cache/huggingface/hub/models--openai-community--gpt2-xl/snapshots/15ea56dee5df4983c59b2538573817e1667135e2/"
bin_files = glob(os.path.join(input, '*.bin'))

print(bin_files)


In [ ]:
# Load model directly
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")

In [ ]:
from transformers import OPTConfig, OPTModel

# Initializing a OPT facebook/opt-large style configuration
configuration = OPTConfig()
print(configuration)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("/users/rh/.cache/modelscope/hub/models/skyline2006/llama-7b", use_fast=False)
# 假设 tokenizer 是你当前使用的 tokenizer 对象
print(tokenizer.unk_token)          # 查看当前 unk_token 是什么
print(tokenizer.unk_token_id)       # 这行可能也会触发递归，谨慎运行

# 如果 unk_token 为 None 或不在词汇表中，手动设置一个合理的值
if tokenizer.unk_token is None:
    # 对于大多数预训练模型，常用的 unk_token 是 "[UNK]" 或 "<unk>"
    tokenizer.unk_token = "[UNK]"
    # 确保该 token 在词汇表中，若不在则添加
    if tokenizer.unk_token not in tokenizer.get_vocab():
        tokenizer.add_tokens([tokenizer.unk_token])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ============== 【必填】修改为你的本地模型文件夹路径 ==============
# 就是你转换后生成 pytorch_model-xx.bin 的文件夹
MODEL_PATH = "anonymous4chan/llama-2-7b" 
# =================================================================

# 1. 加载分词器 + 模型（自动加载bin格式，离线运行）
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,  # 显卡用float16，CPU用float32
    device_map="auto",           # 自动分配GPU/CPU
    low_cpu_mem_usage=True       # 节约内存
)

# 修复LLaMA分词器无pad_token的问题（必加，否则报错）
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

# 2. 处理输入文本（核心：转为模型可识别的张量）
text = "你好"
inputs = tokenizer(
    text,
    return_tensors="pt",  # 转为PyTorch张量
    truncation=True
).to(model.device)  # 自动移到GPU/CPU

# 3. 【极简验证】模型前向推理（不生成文本，仅测试能否运行）
print("=== 开始验证模型推理 ===")
with torch.no_grad():  # 关闭梯度计算，加速推理
    outputs = model(**inputs)

# 4. 输出结果：不报错=推理正常！
print("✅ 模型推理验证成功！")
print(f"输入文本：{text}")
print(f"模型输出张量形状：{outputs.logits.shape}")

In [ ]:
from transformers import AutoModelForCausalLM, GenerationConfig

# ===================== 只需修改这2个路径 =====================
# 1. 你缓存里的 safetensors 模型路径
INPUT_PATH = r"/users/rh/.cache/huggingface/hub/models--anonymous4chan--llama-2-7b/snapshots/708ab4b8207d198cab5715f8de117b7b80c683da"
# 2. 转换后 bin 模型的保存路径
OUTPUT_PATH = r"/users/rh/.cache/huggingface/hub/models--anonymous4chan--llama-2-7b/snapshots/708ab4b8207d198cab5715f8de117b7b80c683da"
# ===========================================================

# 加载 safetensors 模型
model = AutoModelForCausalLM.from_pretrained(
    INPUT_PATH,
    torch_dtype="auto",
    device_map="cpu"  # 用CPU转换，不占显存
)
model.generation_config = GenerationConfig()
# ✅ 关键：保存为 .bin 格式（自动生成 pytorch_model-xx.bin）
model.save_pretrained(
    OUTPUT_PATH,
    safe_serialization=False  # 关闭 safetensors，强制输出 bin
)

print(f"转换完成！.bin 文件保存在：{OUTPUT_PATH}")

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("anonymous4chan/llama-2-7b")

print(config)

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("/users/rh/.cache/modelscope/hub/models/LLM-Research/Llama-3.2-1B/converted_bin_v2")
print(config)


In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from transformers import Qwen2Config

# 加载配置
config = Qwen2Config()
print(config)

In [ ]:
!pip show numpy | head -n 10

In [ ]:
import random 
import dataclasses
from typing import List
import marshal

@dataclasses.dataclass
class TestRequest:
    """
    TestRequest: A request for testing the server's performance
    """
    
    prompt: str
    prompt_len: int
    output_len: int
    
@dataclasses.dataclass
class Dataset:
    """
    Dataset: A dataset for testing the server's performance
    """
 
    dataset_name: str	# "sharegpt" / "alpaca" / ...
    reqs: List[TestRequest]
    
    def dump(self, output_path: str):
        marshal.dump({
            "dataset_name": self.dataset_name,
            "reqs": [(req.prompt, req.prompt_len, req.output_len) for req in self.reqs]
        }, open(output_path, "wb"))
    
    @staticmethod
    def load(input_path: str):
        loaded_data = marshal.load(open(input_path, "rb"))
        return Dataset(
            loaded_data["dataset_name"],
            [TestRequest(req[0], req[1], req[2]) for req in loaded_data["reqs"]]
        )

random.seed(0)
dataset = Dataset.load("/users/rh/DistServe/evaluation/2-benchmark-serving/sharegpt.json")
N = 100
test_reqs = random.sample(dataset.reqs, N)
print(test_reqs[0])
